# Уравнение мелкой воды

## Базовый уровень

Решаем уравнение мелкой воды

\begin{equation*}
\left\{
    \begin{aligned}
      &\frac{\partial h}{\partial t} + \frac{\partial \left(hu\right)}{\partial x} = 0,\\
      &\frac{\partial \left(hu\right)}{\partial t} + \frac{\partial \left(h u^2 + \frac{1}{2} g h^2\right)}{\partial x} = 0.
    \end{aligned}
\right.
\end{equation*}

Линеаризуем их следующим образом $h = H + \eta$, $|\eta| \ll H$, $u \ll c$, $c = \sqrt{g H}$, $H \equiv \text{const}$. Опуская члены второго порядка малости получим следующую систему дифференциальных уравнений в частных производных:

\begin{equation*}
\left\{
    \begin{aligned}
        &\frac{\partial \eta}{\partial t} + H \frac{\partial u}{\partial x} = 0,\\
        &\frac{\partial u}{\partial t} + g \frac{\partial \eta}{\partial x} = 0.
    \end{aligned}
\right.
\end{equation*}

Перепишем её в векторном консервативном виде:

\begin{equation*}
\frac{\partial \mathbf U}{\partial t} + \frac{\partial \mathbf{F(\mathbf U)}}{\partial x} = 0.
\end{equation*}

За векторную переменную возьмём $\mathbf U = (\eta, u)^T$. Тогда система перепишется в следующем виде

\begin{equation*}
\frac{\partial \mathbf U}{\partial t} + A \frac{\partial \mathbf U}{\partial x} = 0,
\end{equation*}

где 

\begin{equation*}
A = \begin{pmatrix}
0 & H\\
g & 0
\end{pmatrix},
\quad
\frac{\partial \mathbf F(\mathbf U)}{\partial \mathbf U} = A.
\end{equation*}

Будем решать систему с помощью схемы Лакса-Фридрихса:

\begin{equation*}
\mathbf U^{n+1}_i = \frac{1}{2}\left(\mathbf U^n_{i-1} + \mathbf U^n_{i+1}\right) - \frac{\Delta t}{2 \Delta x}\left(\mathbf F(\mathbf U^n_{i+1}) - \mathbf F(\mathbf U^n_{i-1})\right),
\end{equation*}

где $\mathbf F(\mathbf U) = A \mathbf U$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as anim 

In [ ]:
g = 9.81
H = 1

def flux(U):
    A = np.array([[0, H], [g, 0]])
    return U @ A.T

def lax_friedrichs(U0, dt, timesteps, dx, N):
    U = np.zeros((timesteps+1, N, 2))
    U[0] = U0
    for n in range(timesteps):
        U_left = np.roll(U[n], 1, axis=0)
        U_right = np.roll(U[n], -1, axis=0)
        U[n+1] = 0.5 * (U_left + U_right) - 0.5 * dt / dx * (flux(U_right) - flux(U_left))
    return U

Реализуем распространение горба воды в разные стороны. Граничные условия были выбраны периодическими.

In [ ]:
def initial_conditions(x):
    return np.stack([np.exp(-100*(x - 0.5)**2), np.zeros_like(x)], axis=1)

In [ ]:
N = 100
x = np.linspace(0.0, 1.0, N)
dx = x[1] - x[0]
timesteps = 500
T = 1.0
dt = T / timesteps

U0 = initial_conditions(x)

In [ ]:
U = lax_friedrichs(U0, dt, timesteps, dx, N)

In [ ]:
fig, ax = plt.subplots()
ax.set_xlabel('x')
ax.set_ylabel('eta(t, x)')
ax.grid(True)
ax.set_title('Решение линеаризованного уравнения мелкой воды')

line, = ax.plot(x, U[0, :, 0], c='blue', linewidth=2.0, alpha=0.5)

def animate(i, x, U):
    line.set_data(x, U[i, :, 0])
    return line,

ani = anim.FuncAnimation(fig, animate, frames=timesteps // 5, interval=100, fargs=(x, U[::5],))

#ani.save('shallow_water.gif')

from IPython.display import HTML
HTML(ani.to_jshtml())

## Продвинутый уровень

Реализуем решение полной нелинейной системы уравнений с помощью схемы Лакса-Вендроффа.

\begin{equation*}
\left\{
    \begin{aligned}
      &\frac{\partial h}{\partial t} + \frac{\partial \left(hu\right)}{\partial x} = 0,\\
      &\frac{\partial \left(hu\right)}{\partial t} + \frac{\partial \left(h u^2 + \frac{1}{2} g h^2\right)}{\partial x} = 0.
    \end{aligned}
\right.
\end{equation*}

\begin{equation*}
\left\{
  \begin{aligned}
    &\frac{\partial h}{\partial t} + \frac{\partial \tilde u}{\partial x} = 0,\\
    &\frac{\partial \tilde u}{\partial t} + \frac{\partial \left(\frac{1}{2} g h^2 + \frac{\tilde u^2}{h}\right)}{\partial x} = 0.
  \end{aligned}
\right.
\end{equation*}

Или в консервативной форме:

\begin{equation*}
\frac{\partial \mathbf U}{\partial t} + \frac{\partial \mathbf{F(\mathbf U)}}{\partial x} = 0,
\end{equation*}

где $\mathbf U = \left(h, h u\right)^T = \left(h, \tilde u\right)^T$, $\mathbf F (\mathbf U) = \left(\tilde u, \frac{1}{2} g h^2 + \frac{\tilde u^2}{h}\right)^T$.

Якобиан $A(\mathbf U) = \frac{\partial \mathbf F(\mathbf U)}{\partial \mathbf U}$ выражается следующим образом

\begin{equation*}
A(h, \tilde u) = 
\begin{pmatrix}
0 & 1 \\
g h - \frac{\tilde u^2}{h^2} & \frac{2 \tilde u}{h}
\end{pmatrix}.
\end{equation*}

Решение для системы мелкой воды реализуем с помощью схемы Лакса-Вендроффа:

$$\mathbf U^{n+1}_i = \mathbf U^n_i - \frac{\Delta t}{2 \Delta x} \left[\mathbf F(\mathbf U^n_{i+1}) - \mathbf F(\mathbf U^n_{i-1})\right] + \frac{\Delta t^2}{2 \Delta x^2}\left[A_{i+1/2}\left(\mathbf F(\mathbf U^n_{i+1}) - \mathbf F(\mathbf U^n_i)\right) - A_{i-1/2}\left(\mathbf F(\mathbf U^n_i) - \mathbf F(\mathbf U^n_{i-1})\right)\right],$$

где $A_{i \pm 1/2}$ -- якобиан $\mathbf F(\mathbf U)$ в точке $\frac{1}{2}(\mathbf U^n_i + \mathbf U^n_{i \pm 1})$.

In [ ]:
def flux(U):
    h, hu = U[:, 0], U[:, 1]
    F = np.zeros_like(U)
    F[:, 0] = hu
    F[:, 1] = hu**2 / h + 0.5 * g * h**2
    return F

def jacobian(U):
    h, hu = U[:, 0], U[:, 1]
    A = np.zeros((len(h), 2, 2))
    A[:, 0, 0] = 0.0
    A[:, 0, 1] = 1.0
    A[:, 1, 0] = g * h - (hu**2) / (h**2)
    A[:, 1, 1] = 2 * hu / h
    return A

def lax_wendroff(U0, dt, timesteps, dx, N, periodic_bc=False, wall_bc=False):
    U = np.zeros((timesteps+1, N, 2))
    U[0] = U0

    for n in range(timesteps):
        U_n = U[n].copy()

        if periodic_bc:
            U_plus  = np.roll(U_n, -1, axis=0)
            U_minus = np.roll(U_n, 1, axis=0)
        elif wall_bc:
            U_ext = np.zeros((N+2, 2))
            U_ext[1:-1] = U_n
            U_ext[0, 0]   = U_ext[1, 0]
            U_ext[0, 1]   = -U_ext[1, 1]
            U_ext[-1, 0]  = U_ext[-2, 0]
            U_ext[-1, 1]  = -U_ext[-2, 1]
            U_plus  = U_ext[2:]
            U_minus = U_ext[:-2]
            U_n     = U_ext[1:-1]
        else:
            U_plus  = np.roll(U_n, -1, axis=0)
            U_minus = np.roll(U_n, 1, axis=0)
            U_plus[-1] = U_plus[-2]
            U_minus[0] = U_minus[1]

        F = flux(U_n)
        F_plus = flux(U_plus)
        F_minus = flux(U_minus)
        A_plus  = jacobian(0.5 * (U_n + U_plus))
        A_minus = jacobian(0.5 * (U_n + U_minus))
        U[n+1] = U_n - (dt / (2.0 * dx)) * (F_plus - F_minus) + (dt**2) / (2.0 * dx**2) * (
            np.einsum('nij,nj->ni', A_plus, (F_plus - F)) -
            np.einsum('nij,nj->ni', A_minus, (F - F_minus))
        )

    return U


Зададим начальные условия для задачи о распаде разрыва:

\begin{equation*}
h(0, x) =
\begin{cases}
h_L, \quad x < 0.5\\
h_R, \quad x > 0.5
\end{cases}
, \text{ где } h_L > h_R;
\quad
u(0, x) \equiv 0.
\end{equation*}

In [ ]:
def initial_conditions(x):
    h = np.where(x > 0.5, 1.0, 2.0)
    u = np.zeros_like(x)
    hu = h * u
    return np.stack([h, hu], axis=1)

In [ ]:
N = 500
x = np.linspace(0.0, 1.0, N)
dx = x[1] - x[0]
timesteps = 10000
T = 0.1
dt = T / timesteps

U0 = initial_conditions(x)

In [ ]:
c = np.sqrt(g * np.max(U0[:, 0]))
CFL = c * dt / dx
print("CFL =", CFL)

In [ ]:
U = lax_wendroff(U0, dt, timesteps, dx, N, wall_bc = True)

In [ ]:
fig, ax = plt.subplots()
ax.set_xlabel('x')
ax.set_ylabel('h(t, x)')
ax.grid(True)
ax.set_title('Решение уравнения мелкой воды для задачи о распаде разрыва')

line, = ax.plot(x, U[0, :, 0], c='blue', linewidth=2.0, alpha=0.5)

def animate(i, x, U):
    line.set_data(x, U[i, :, 0])
    return line,

ani = anim.FuncAnimation(fig, animate, frames=timesteps // 100, interval=100, fargs=(x, U[::100],))

#ani.save('shallow_water_nonlinear.gif')

from IPython.display import HTML
HTML(ani.to_jshtml())

На анимации видно формирование ударной волны, идущей вправо, и волны разрежения, идущей влево. Сильно проявляется численная дисперсия, присущая схеме Лакса-Вендроффа.